# 02-1 · The feasible set F

| | |
|---|---|
| **What it does** | drops every restaurant where ANY member has nothing safe to eat, before any scoring happens |
| **Proves** | choosing only from F makes a safety violation STRUCTURALLY impossible — not a matter of prompt quality |
| **Source** | original to this paper (`potluck/backend/services/aggregation.py`, `dish_is_safe`/`build_feasible_set`), harvested into `nbio.py` |
| **Guardrail** | `dish_is_safe` — checked directly below, including its known failure mode |
| **Eval surface** | flip one member's allergy and watch the feasible set shrink |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/research/ai-in-food/fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/research/ai-in-food/fair-mediator')

## The guardrail, read directly

```python
def dish_is_safe(dish_name, diet, allergies):
    text = (dish_name or "").lower()
    for allergen in allergies or []:
        if allergen and allergen.lower() in text:
            return False
    diet = (diet or "mix").lower()
    if diet in {"veg", "vegetarian"}:
        return not is_nonveg_text(text)
    if diet == "vegan":
        return not is_nonveg_text(text) and not any(w in text for w in NON_VEGAN_WORDS)
    return True
```
`nbio.py` (this repo). It checks the dish NAME against keyword lists — nothing more.


In [3]:
# Step 1: probe the guardrail on cases that matter
probes = [
    ("Paneer Butter Masala", "veg", []),
    ("Paneer Butter Masala", "vegan", []),
    ("Chicken Biryani", "veg", []),
    ("Chicken Biryani", "non-veg", []),
    ("Peanut Chikki", "mix", ["peanuts"]),
    ("Crispy Baby Corn", "veg", ["gluten"]),   # the failure mode
]
nbio.table([{"dish": d, "diet": diet, "allergies": str(a), "safe": nbio.dish_is_safe(d, diet, a)}
            for d, diet, a in probes], cols=["dish", "diet", "allergies", "safe"], width=24)


dish                  diet     allergies    safe 
-------------------------------------------------
Paneer Butter Masala  veg      []           True 
Paneer Butter Masala  vegan    []           False
Chicken Biryani       veg      []           False
Chicken Biryani       non-veg  []           True 
Peanut Chikki         mix      ['peanuts']  True 
Crispy Baby Corn      veg      ['gluten']   True 


> **The last row is the known weakness, demonstrated rather than described.**
> "Crispy Baby Corn" is battered and fried — almost certainly containing gluten —
> but the filter only rejects a dish whose NAME contains the allergen string, so
> it passes for a coeliac diner. This is why **PRD-4** proposes an offline
> ingredient-inference pass rather than trusting dish names.


## Build the feasible set on real data


In [4]:
# Step 2: load the group (4 real members from the harvested fixture) and the catalog
personas = json.load(open(nbio.DATA / "personas.json"))
group_names = ["Vihaan", "Vincent", "Vasu", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"],
                     "taste": p["taste_vector"]})
for m in members:
    print(f"   {m['display_name']:<8} diet={m['diet']:<8} allergies={m['allergies']}")


   Vihaan   diet=veg      allergies=[]
   Vincent  diet=vegan    allergies=['dairy']
   Vasu     diet=mix      allergies=[]
   Alice    diet=non-veg  allergies=['peanuts']


In [5]:
# Step 3: build a tiny catalog from the real menus, then the feasible set
def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip()
            return
        for v in node.values():
            yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node:
            yield from iter_dish_names(v)

catalog = []
for f in sorted((nbio.DATA / "Menu").glob("*.json")):
    payload = json.loads(f.read_text())
    dishes = [n for n in iter_dish_names(payload) if n]
    catalog.append({"option_id": f.stem, "name": f.stem, "menu": [{"name": d} for d in dishes]})

before = {"catalog": len(catalog)}
F = nbio.build_feasible_set(catalog, members)
after = {"catalog": len(catalog), "F": len(F)}
nbio.io(1, "build_feasible_set", before, after, reads=["catalog", "members(diet, allergies)"])

for o in F:
    counts = {m["display_name"][:7]: len(o["safe_dishes"][m["member_id"]]) for m in members}
    print(f"   KEEP {o['name'][:34]:<36} {counts}")


+- STEP 1  build_feasible_set                                       [py]
| IN      : catalog, members(diet, allergies)
| NEW     : F
| CHANGED : -
+-------------------------------------------------------------------------
   KEEP Arabian Corner Restaurant - Skylou   {'Vihaan': 23, 'Vincent': 15, 'Vasu': 60, 'Alice': 60}
   KEEP Casa.cafe                            {'Vihaan': 70, 'Vincent': 44, 'Vasu': 129, 'Alice': 129}
   KEEP Delicia Multi Cuisine Restaurant     {'Vihaan': 94, 'Vincent': 67, 'Vasu': 155, 'Alice': 155}
   KEEP House of Chefs                       {'Vihaan': 123, 'Vincent': 88, 'Vasu': 198, 'Alice': 198}
   KEEP Ministry of South Bar and Kitchen    {'Vihaan': 20, 'Vincent': 15, 'Vasu': 32, 'Alice': 32}


## The property, checked rather than trusted

If F is correct, every member has at least one safe dish at every option inside
it, and no dish in any safe set fails a re-check. Assert it.


In [6]:
# Step 4: assert the safety property that the paper's Methods section claims
violations = []
for o in F:
    for m in members:
        safe = o["safe_dishes"][m["member_id"]]
        if not safe:
            violations.append((o["name"], m["display_name"], "empty safe set"))
        for d in safe:
            if not nbio.dish_is_safe(d["name"], m["diet"], m["allergies"]):
                violations.append((o["name"], m["display_name"], d["name"]))

print("safety violations inside F:", len(violations))
assert not violations, violations[:5]
print("PASS -- every member has >=1 safe dish at every option in F,")
print("        and no dish in any safe set fails a re-check.")


safety violations inside F: 0
PASS -- every member has >=1 safe dish at every option in F,
        and no dish in any safe set fails a re-check.


### Reading it

The assertion above is the paper's core structural claim, made checkable: **a
constraint violation cannot occur downstream of this cell**, because F was built
by removing every option that would allow one. Contrast this with **05-mediate**,
where the LLM chat path receives the same constraints as English prose instead
of a structural filter — and violates them.

Next: **03-aggregate** scores every member against every option remaining in F.
